# 첫번째와 두 번째 셀에 있는 crop_name 과 item_code를 바꾸고 사용하시면 됩니다.
## 결측치가 있는 행은 제거 됩니다. (ex: 2018년 1~4주차는 lag 데이터가 없어서 제거됨)
### 아이템 코드
- 양파 : 1201
- 배추 : 1001
- 상추 : 1005
- 사과 : 0601
- 무 : 1101
- 감자 : 0501
- 대파 : 1202
- 건고추 : 1207
- 마늘 : 1209
- 딸기 : 0804
- 방울토마토 : 0806
- 오이 : 0901
- 양배추 : 1004
- 고구마 : 0502
- 배 : 0602

In [1]:
import pandas as pd

# ✅ 작물 이름 바꾸기
crop_name = '배추'  # 여기만 '사과', '무', '양파' 등으로 바꾸면 됨

# ✅ 파일 경로 자동 생성
file_path = f"data/{crop_name}_이상치제거_주간기준_등급코드.csv"

# 파일 로드
df = pd.read_csv(file_path, encoding='cp949')
df_grow = pd.read_csv('data/factor_external_weekly.csv', encoding='utf-8') # 이건 고정
# 이후 df를 기반으로 전처리, 병합, lag 생성 등 전체 코드 실행


In [2]:
# ✅ 작물 코드 바꾸기
df['item_code'] = '1001'  # 거래데이터에는 대분류 코드가 없고 한 종류만 있음

df_grow['item_code'] = df_grow['item_code'].astype(str) 
df_grow['item_code'] = df_grow['item_code'].str.zfill(4)

In [3]:
# weekno 열을 만들고 주차 표기를 통일시켜 merge 준비
def get_week_of_year(date):
    date = pd.to_datetime(date)
    year = date.year
    week_number = date.isocalendar().week
    return f"{year}{week_number:02d}"

df['weekno'] = df['연월일'].apply(get_week_of_year)

In [4]:
def format_week_int(week_no):
    # week_no가 202401, 202402 등 정수형이면
    week_no = int(week_no)
    year = week_no // 100
    week = week_no % 100
    return f"{year}{week:02d}"

df_grow['weekno'] = df_grow['week_no'].apply(format_week_int)

In [5]:
# 기 생성된 휴일여부	명절지수	작기정보 칼럼 삭제
df.drop(columns=['휴일여부', '명절지수', '작기정보'], inplace=True)

In [6]:
# 병합하기
merged_df  = pd.merge( df,
                    df_grow[['weekno', 'item_code', 'holiday_flag', 'holiday_score', 'grow_score']],
                    left_on=['weekno', 'item_code'],
                    right_on=['weekno', 'item_code'],
                    how='left'
                )
merged_df.head()

,주차,연월일,품목코드,품목명,품종코드,품종명,등급코드,등급이름,총금액(원),총거래량(kg),...,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),item_code,weekno,holiday_flag,holiday_score,grow_score
0,2018-01-01~2018-01-07,2018-01-03,1,배추,99,기타배추,13,저,11004000.0,25860.0,...,3.7,-1.7,56.9,-9.0,-9.0,1001,201801,3,0.0,7.0
1,2018-01-01~2018-01-07,2018-01-03,1,배추,10,배추뿌리,12,중,296800.0,320.0,...,2.2,-4.8,63.0,-9.0,-9.0,1001,201801,3,0.0,7.0
2,2018-01-01~2018-01-07,2018-01-03,1,배추,0,배추,13,저,849200.0,1800.0,...,-0.2,-99.0,-9.0,-9.0,-9.0,1001,201801,3,0.0,7.0
3,2018-01-01~2018-01-07,2018-01-03,1,배추,8,쌈배추,11,고,118500.0,80.0,...,-0.6,-12.5,53.0,-9.0,-9.0,1001,201801,3,0.0,7.0
4,2018-01-01~2018-01-07,2018-01-03,1,배추,9,우거지,11,고,38340.0,30.0,...,-0.4,-7.1,42.3,-9.0,-9.0,1001,201801,3,0.0,7.0


In [7]:
# 강수량, 1시간최고 강수량은 결측치(-9) 혹은 비가 안옴(0)이 많아 0 이하는 0으로 처리
# merged_df['강수량(mm)'] = merged_df['강수량(mm)'<=0].count()
merged_df.loc[merged_df['강수량(mm)']<=0, '강수량(mm)'] = 0
merged_df.loc[merged_df['1시간최고강수량(mm)']<=0, '1시간최고강수량(mm)'] = 0

In [8]:
# 일간 거래 데이터 (필요한 열만 사용)
df_daily_galic = merged_df.loc[:, ['연월일', '품종코드', '등급코드', '총거래량(kg)','주간평균단가(원)','직팜산지코드','일평균기온','최고기온','최저기온','평균상대습도','강수량(mm)','1시간최고강수량(mm)', 'holiday_flag', 'holiday_score','grow_score', '총금액(원)']]
df_daily_galic.head()

,연월일,품종코드,등급코드,총거래량(kg),주간평균단가(원),직팜산지코드,일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,총금액(원)
0,2018-01-03,99,13,25860.0,735.410074,1094,0.2,3.7,-1.7,56.9,0.0,0.0,3,0.0,7.0,11004000.0
1,2018-01-03,10,12,320.0,735.410074,1109,-1.3,2.2,-4.8,63.0,0.0,0.0,3,0.0,7.0,296800.0
2,2018-01-03,0,13,1800.0,735.410074,1034,-99.0,-0.2,-99.0,-9.0,0.0,0.0,3,0.0,7.0,849200.0
3,2018-01-03,8,11,80.0,735.410074,1045,-6.5,-0.6,-12.5,53.0,0.0,0.0,3,0.0,7.0,118500.0
4,2018-01-03,9,11,30.0,735.410074,1000,-4.7,-0.4,-7.1,42.3,0.0,0.0,3,0.0,7.0,38340.0


In [9]:
# 주간 거래 데이터 병합
df_galic = merged_df.loc[:, ['weekno', '품종코드', '등급코드', '총금액(원)', '총거래량(kg)','직팜산지코드','일평균기온','최고기온','최저기온','평균상대습도','강수량(mm)','1시간최고강수량(mm)', 'holiday_flag', 'holiday_score','grow_score']]
df_weekly_galic = df_galic.groupby(['weekno', '품종코드', '등급코드', '직팜산지코드']).agg({
    '총금액(원)' : 'sum', 
    '총거래량(kg)' : 'sum', 
    '일평균기온': 'mean', 
    '최고기온': 'max', 
    '최저기온' : 'min', 
    '평균상대습도' : 'mean', 
    '강수량(mm)' : 'sum', 
    '1시간최고강수량(mm)' : 'sum', 
    'holiday_flag' : 'sum', 
    'holiday_score' : 'sum', 
    'grow_score' : 'sum'
}).reset_index()

In [10]:
import pandas as pd
from datetime import date  # 여기서 date만 가져옴

# ✅ 주간 단가 계산
df_weekly_galic['평균단가(원)'] = round(df_weekly_galic['총금액(원)'] / df_weekly_galic['총거래량(kg)'])

# ✅ weekno에서 year, week 분리
df_weekly_galic['year'] = df_weekly_galic['weekno'].astype(str).str[:4].astype(int)
df_weekly_galic['week'] = df_weekly_galic['weekno'].astype(str).str[4:].astype(int)

# ✅ 연도별 최대 주차 수
def get_max_week(year):
    return date(year, 12, 28).isocalendar()[1]

# ✅ 유효한 주차만 필터링
df_weekly_galic = df_weekly_galic[df_weekly_galic.apply(
    lambda row: 1 <= row['week'] <= get_max_week(row['year']), axis=1
)].copy()

# ✅ 주차 → 주간 시작일로 변환
def year_week_to_date(year, week):
    return date.fromisocalendar(year, week, 1)  # ← 여기도 date

df_weekly_galic['week_start'] = df_weekly_galic.apply(
    lambda row: year_week_to_date(row['year'], row['week']),
    axis=1
)
df_weekly_galic['week_start'] = pd.to_datetime(df_weekly_galic['week_start'])

# ✅ weekno 제거
df_weekly_galic.drop(columns='weekno', inplace=True)


In [11]:
# # 주간 단가 계산
# df_weekly_galic['평균단가(원)'] = round(df_weekly_galic['총금액(원)'] / df_weekly_galic['총거래량(kg)'])

# # weekno에서 year, week 분리 (문자열 슬라이싱)
# df_weekly_galic['year'] = df_weekly_galic['weekno'].astype(str).str[:4]
# df_weekly_galic['week'] = df_weekly_galic['weekno'].astype(str).str[4:]

# # 주간 시작일 입력 (시계열 특성 - prophet)
# import datetime

# def year_week_to_date(year, week):
#     # ISO 주차는 매년 첫 번째 주의 월요일이 기준
#     return datetime.date.fromisocalendar(int(year), int(week), 1)  # 1: 월요일

# df_weekly_galic['week_start'] = df_weekly_galic.apply(lambda row: year_week_to_date(row['year'], row['week']), axis=1)
# df_weekly_galic['week_start'] = pd.to_datetime(df_weekly_galic['week_start'])
# df_weekly_galic.drop(columns='weekno', inplace=True)

In [12]:
# 저장
# df_daily_galic.to_csv('data/trade_daily_galic.csv', encoding='cp949', index=False)
df_weekly_galic.to_csv('data/trade_weekly_galic.csv', encoding='cp949', index=False)

# 샘플파일 생성 (chat과 편안한 상담용 )
# df_daily_galic_sample = df_daily_galic.iloc[:100]
# df_daily_galic_sample.to_csv('data/trade_daily_galic_sample.csv', encoding='cp949', index=False)
# df_weekly_galic_sample = df_weekly_galic.iloc[:100]
# df_weekly_galic_sample.to_csv('data/trade_weekly_galic_sample.csv', encoding='cp949', index=False)

In [13]:
df_weekly_galic

,품종코드,등급코드,직팜산지코드,총금액(원),총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,평균단가(원),year,week,week_start
0,0,12,1079,96000.0,160.0,1.400000,5.2,-1.2,58.900000,0.0,0.0,3,0.0,7.0,600.0,2018,1,2018-01-01
1,0,12,1094,10624000.0,19260.0,0.233333,4.2,-1.9,58.500000,0.0,0.0,9,0.0,21.0,552.0,2018,1,2018-01-01
2,0,12,1120,1703400.0,2988.0,1.800000,5.6,-1.3,27.000000,0.0,0.0,6,0.0,14.0,570.0,2018,1,2018-01-01
3,0,13,1001,3457800.0,9820.0,-6.100000,-0.8,-11.7,68.800000,0.0,0.0,3,0.0,7.0,352.0,2018,1,2018-01-01
4,0,13,1034,1149700.0,2640.0,-50.900000,3.2,-99.0,31.150000,0.0,0.0,6,0.0,14.0,435.0,2018,1,2018-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148792,99,13,1159,168000.0,420.0,18.200000,23.3,14.3,77.000000,0.4,0.4,2,0.0,0.0,400.0,2025,22,2025-05-26
148793,99,13,1164,26284200.0,59640.0,19.383333,27.4,11.0,66.583333,0.0,0.0,12,0.0,0.0,441.0,2025,22,2025-05-26
148794,99,13,1165,3950000.0,8690.0,20.900000,27.0,15.1,51.800000,0.0,0.0,2,0.0,0.0,455.0,2025,22,2025-05-26
148795,99,13,1166,4172700.0,15240.0,18.900000,26.9,13.9,68.066667,0.0,0.0,6,0.0,0.0,274.0,2025,22,2025-05-26


In [14]:
df = pd.concat([df_weekly_galic.drop(columns=['평균단가(원)', '총금액(원)']), df_weekly_galic.iloc[:, -4]], axis=1)
# df = pd.concat([df_weekly_galic_sample.drop(columns=['평균단가(원)', '총금액(원)']), df_weekly_galic_sample.iloc[:, -4]], axis=1)

In [15]:
df.to_csv('data/trade_weekly_galic.csv', encoding='cp949', index=False)

# 모델링

In [16]:
import pandas as pd
import numpy as np

In [17]:
df_weekly_galic = pd.read_csv('data/trade_weekly_galic.csv', encoding='cp949')
# df_weekly_galic_sample = pd.read_csv('data/trade_weekly_galic_sample.csv', encoding='cp949')

In [18]:
print(df_weekly_galic.isna().sum())  # 대체로 산지가 없거나 수입산인 경우 기후 데이터 없음
df_weekly_galic_drop = df_weekly_galic.dropna()

품종코드               0
등급코드               0
직팜산지코드             0
총거래량(kg)           0
일평균기온            614
최고기온             614
최저기온             614
평균상대습도           614
강수량(mm)            0
1시간최고강수량(mm)       0
holiday_flag       0
holiday_score      0
grow_score         0
year               0
week               0
week_start         0
평균단가(원)            0
dtype: int64


In [19]:
df_weekly_galic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148773 entries, 0 to 148772
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   품종코드           148773 non-null  int64  
 1   등급코드           148773 non-null  int64  
 2   직팜산지코드         148773 non-null  int64  
 3   총거래량(kg)       148773 non-null  float64
 4   일평균기온          148159 non-null  float64
 5   최고기온           148159 non-null  float64
 6   최저기온           148159 non-null  float64
 7   평균상대습도         148159 non-null  float64
 8   강수량(mm)        148773 non-null  float64
 9   1시간최고강수량(mm)   148773 non-null  float64
 10  holiday_flag   148773 non-null  int64  
 11  holiday_score  148773 non-null  float64
 12  grow_score     148773 non-null  float64
 13  year           148773 non-null  int64  
 14  week           148773 non-null  int64  
 15  week_start     148773 non-null  object 
 16  평균단가(원)        148773 non-null  float64
dtypes: float64(10), int64(6), obj

In [20]:
# 라이브러리 임포트
import pandas as pd
import numpy as np

In [21]:
# 데이터 로드
df = pd.read_csv('data/trade_weekly_galic.csv', encoding='cp949', parse_dates=['week_start'])
print(df.shape)
df.head()

(148773, 17)


,품종코드,등급코드,직팜산지코드,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,year,week,week_start,평균단가(원)
0,0,12,1079,160.0,1.400000,5.2,-1.2,58.90,0.0,0.0,3,0.0,7.0,2018,1,2018-01-01,600.0
1,0,12,1094,19260.0,0.233333,4.2,-1.9,58.50,0.0,0.0,9,0.0,21.0,2018,1,2018-01-01,552.0
2,0,12,1120,2988.0,1.800000,5.6,-1.3,27.00,0.0,0.0,6,0.0,14.0,2018,1,2018-01-01,570.0
3,0,13,1001,9820.0,-6.100000,-0.8,-11.7,68.80,0.0,0.0,3,0.0,7.0,2018,1,2018-01-01,352.0
4,0,13,1034,2640.0,-50.900000,3.2,-99.0,31.15,0.0,0.0,6,0.0,14.0,2018,1,2018-01-01,435.0


In [22]:
# 결측치 처리 (간단히 결측치 행 제거)
df = df.dropna()
print(df.isna().sum())

품종코드             0
등급코드             0
직팜산지코드           0
총거래량(kg)         0
일평균기온            0
최고기온             0
최저기온             0
평균상대습도           0
강수량(mm)          0
1시간최고강수량(mm)     0
holiday_flag     0
holiday_score    0
grow_score       0
year             0
week             0
week_start       0
평균단가(원)          0
dtype: int64


In [23]:
df.head()

,품종코드,등급코드,직팜산지코드,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,year,week,week_start,평균단가(원)
0,0,12,1079,160.0,1.400000,5.2,-1.2,58.90,0.0,0.0,3,0.0,7.0,2018,1,2018-01-01,600.0
1,0,12,1094,19260.0,0.233333,4.2,-1.9,58.50,0.0,0.0,9,0.0,21.0,2018,1,2018-01-01,552.0
2,0,12,1120,2988.0,1.800000,5.6,-1.3,27.00,0.0,0.0,6,0.0,14.0,2018,1,2018-01-01,570.0
3,0,13,1001,9820.0,-6.100000,-0.8,-11.7,68.80,0.0,0.0,3,0.0,7.0,2018,1,2018-01-01,352.0
4,0,13,1034,2640.0,-50.900000,3.2,-99.0,31.15,0.0,0.0,6,0.0,14.0,2018,1,2018-01-01,435.0


In [24]:
# year, week, week_start 컬럼을 제일 앞으로 이동
front_cols = ['year', 'week', 'week_start']
other_cols = [col for col in df.columns if col not in front_cols]
df = df[front_cols + other_cols]
df.reset_index(drop=True, inplace=True)
df

,year,week,week_start,품종코드,등급코드,직팜산지코드,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,평균단가(원)
0,2018,1,2018-01-01,0,12,1079,160.0,1.400000,5.2,-1.2,58.900000,0.0,0.0,3,0.0,7.0,600.0
1,2018,1,2018-01-01,0,12,1094,19260.0,0.233333,4.2,-1.9,58.500000,0.0,0.0,9,0.0,21.0,552.0
2,2018,1,2018-01-01,0,12,1120,2988.0,1.800000,5.6,-1.3,27.000000,0.0,0.0,6,0.0,14.0,570.0
3,2018,1,2018-01-01,0,13,1001,9820.0,-6.100000,-0.8,-11.7,68.800000,0.0,0.0,3,0.0,7.0,352.0
4,2018,1,2018-01-01,0,13,1034,2640.0,-50.900000,3.2,-99.0,31.150000,0.0,0.0,6,0.0,14.0,435.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148154,2025,22,2025-05-26,99,13,1153,1877.0,16.166667,26.7,7.5,73.133333,0.8,0.8,6,0.0,0.0,552.0
148155,2025,22,2025-05-26,99,13,1159,420.0,18.200000,23.3,14.3,77.000000,0.4,0.4,2,0.0,0.0,400.0
148156,2025,22,2025-05-26,99,13,1164,59640.0,19.383333,27.4,11.0,66.583333,0.0,0.0,12,0.0,0.0,441.0
148157,2025,22,2025-05-26,99,13,1165,8690.0,20.900000,27.0,15.1,51.800000,0.0,0.0,2,0.0,0.0,455.0


In [25]:
# 1. 품종 비율 및 누적비율 계산
item_counts = df['품종코드'].value_counts(normalize=True).reset_index()
item_counts.columns = ['품종코드', '비율']
item_counts['누적비율'] = item_counts['비율'].cumsum()

# 2. 누적비율 80% 이하 품종코드만 선택
main_items = item_counts[item_counts['누적비율'] <= 0.8]['품종코드']

# 3. 기타 품종은 숫자 100으로 통합
df['품종코드_통합'] = df['품종코드'].apply(lambda x: x if x in main_items.values else 100)

# 혹시 중복된 '품종코드' 컬럼이 있을 수 있으니 정리
df = df.loc[:, ~df.columns.duplicated()]

# 품종코드 값을 통합된 값으로 덮어쓰기
df['품종코드'] = df['품종코드_통합']

# 통합 컬럼 삭제
df.drop(columns=['품종코드_통합'], inplace=True)




In [26]:
df['품종코드'].unique()

array([100,   1,   4,   8,  99], dtype=int64)

In [27]:
import pandas as pd

# 1. 주차 기준 키 생성
df['yearweek'] = df['year'] * 100 + df['week']

# 2. 기후 컬럼 정의
climate_cols = [
    '일평균기온', '최고기온', '최저기온',
    '평균상대습도', '강수량(mm)', '1시간최고강수량(mm)'
]

# 3. 등급 포함 지연기후용 테이블 생성
climate_df = (
    df[['직팜산지코드', '등급코드', 'year', 'week'] + climate_cols]
    .drop_duplicates(subset=['직팜산지코드', '등급코드', 'year', 'week'])
    .copy()
)
climate_df['yearweek'] = climate_df['year'] * 100 + climate_df['week']

# 4. 지연 변수 생성 (등급 포함 그룹핑)
climate_df.sort_values(['직팜산지코드', '등급코드', 'yearweek'], inplace=True)
for lag in [1, 2, 3, 4]:
    for col in climate_cols:
        climate_df[f'{col}_t-{lag}'] = (
            climate_df.groupby(['직팜산지코드', '등급코드'])[col].shift(lag)
        )

# 5. 지연기후만 남기기
climate_df = climate_df.drop(columns=climate_cols)

# 6. 병합 대상: 원본 df 전체
df['yearweek'] = df['year'] * 100 + df['week']
climate_df['yearweek'] = climate_df['yearweek'].astype(int)

# 7. 병합을 위한 키: 직팜산지코드 + 등급코드 + yearweek
df_final = pd.merge(
    df,
    climate_df,
    how='left',
    on=['직팜산지코드', '등급코드', 'yearweek']
)

# 8. yearweek 제거
df_final.drop(columns='yearweek', inplace=True)

# 9. 컬럼 정리 (필요 시)
df_final.rename(columns={
    'year_x': 'year',
    'week_x': 'week'
}, inplace=True)

df_final.drop(columns=['year_y', 'week_y'], errors='ignore', inplace=True)

# 10. 결과 확인
print("✅ 등급 포함 지연기후 병합 완료")
print("전체 행:", len(df_final))
print("지연기후 샘플:\n", df_final[[col for col in df_final.columns if '_t-' in col]].head())


✅ 등급 포함 지연기후 병합 완료
전체 행: 148159
지연기후 샘플:
    일평균기온_t-1  최고기온_t-1  최저기온_t-1  평균상대습도_t-1  강수량(mm)_t-1  1시간최고강수량(mm)_t-1  \
0        NaN       NaN       NaN         NaN          NaN               NaN   
1        NaN       NaN       NaN         NaN          NaN               NaN   
2        NaN       NaN       NaN         NaN          NaN               NaN   
3        NaN       NaN       NaN         NaN          NaN               NaN   
4        NaN       NaN       NaN         NaN          NaN               NaN   

   일평균기온_t-2  최고기온_t-2  최저기온_t-2  평균상대습도_t-2  ...  최저기온_t-3  평균상대습도_t-3  \
0        NaN       NaN       NaN         NaN  ...       NaN         NaN   
1        NaN       NaN       NaN         NaN  ...       NaN         NaN   
2        NaN       NaN       NaN         NaN  ...       NaN         NaN   
3        NaN       NaN       NaN         NaN  ...       NaN         NaN   
4        NaN       NaN       NaN         NaN  ...       NaN         NaN   

   강수량(mm)_t-3  1시간최고강수량(mm)_t-3

In [28]:
df_final.rename(columns={
    'year_x': 'year',
    'week_x': 'week'
}, inplace=True)

# 확인
df_final[['year', 'week', '직팜산지코드'] + [col for col in df_final.columns if '_t-' in col]].head()



,year,week,직팜산지코드,일평균기온_t-1,최고기온_t-1,최저기온_t-1,평균상대습도_t-1,강수량(mm)_t-1,1시간최고강수량(mm)_t-1,일평균기온_t-2,...,최저기온_t-3,평균상대습도_t-3,강수량(mm)_t-3,1시간최고강수량(mm)_t-3,일평균기온_t-4,최고기온_t-4,최저기온_t-4,평균상대습도_t-4,강수량(mm)_t-4,1시간최고강수량(mm)_t-4
0,2018,1,1079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,1,1094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018,1,1120,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018,1,1001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018,1,1034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# 1. NaN 제거
df_final.dropna(inplace=True)

# 2. 총거래량이 0 이하인 행 제거
df_final = df_final[df_final['총거래량(kg)'] > 0]

# 3. 결과 확인
print("✅ 결측치 및 거래량 0 제거 완료")
print("남은 행 수:", len(df_final))


✅ 결측치 및 거래량 0 제거 완료
남은 행 수: 145430


In [30]:
# 7. CSV 저장

df_final.to_csv(f"data/{crop_name}_lag처리.csv", index=False, encoding='cp949')


print(f"✅ 저장 완료")

✅ 저장 완료


# 정확도 검증

In [31]:
# # 1. yearweek 계산
# df_final['yearweek'] = df_final['year'] * 100 + df_final['week']
# df_final['yearweek_t-1'] = df_final['yearweek'] - 1

# # 2. 기후 참조 테이블 생성 (실제 t-1 확인용)
# climate_check = df_final[['직팜산지코드', '등급코드', 'yearweek', '일평균기온']].drop_duplicates()
# climate_check.rename(columns={
#     'yearweek': 'yearweek_t-1',
#     '일평균기온': '실제_t-1_기온'
# }, inplace=True)

# # 3. 병합
# df_check = pd.merge(
#     df_final,
#     climate_check,
#     how='left',
#     on=['직팜산지코드', '등급코드', 'yearweek_t-1']
# )

# # 4. 비교 및 출력
# df_check['검증결과'] = df_check['일평균기온_t-1'].round(2) == df_check['실제_t-1_기온'].round(2)

# # 5. 결과 샘플 출력
# print("✅ 검증 결과 샘플")
# display(df_check[['직팜산지코드', '등급코드', 'year', 'week', '일평균기온_t-1', '실제_t-1_기온', '검증결과']].head(10))

# # 6. 검증 통계
# print("\n✅ 전체 일치율: {:.2f}%".format(df_check['검증결과'].mean() * 100))
# print("불일치 수:", (~df_check['검증결과']).sum())


In [32]:
# # NaN이 아닌 비교만 남김
# valid_check = df_check[
#     df_check['일평균기온_t-1'].notna() & df_check['실제_t-1_기온'].notna()
# ].copy()

# # 정확도 재계산
# 정확도 = (valid_check['일평균기온_t-1'].round(2) == valid_check['실제_t-1_기온'].round(2)).mean()
# print(f"🔍 유효한 비교 대상 중 정확도: {정확도*100:.2f}%")


In [33]:
# missing_lag = df_check[df_check['일평균기온_t-1'].isna()]
# missing_summary = missing_lag.groupby(['직팜산지코드', '등급코드']).size().sort_values(ascending=False).head(10)
# print("🚫 lag 누락 상위 조합 (직팜산지코드, 등급코드):")
# print(missing_summary)


In [34]:
# # 병합 후 컬럼명이 year_x, week_x 라면 이름 정리
# merged_check.rename(columns={
#     'year_x': 'year',
#     'week_x': 'week'
# }, inplace=True)
